In [ ]:
import pandas as pd
import os
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import norm
import tensorflow as tf
from darts.utils.statistics import check_seasonality
from darts import TimeSeries
from plotly.subplots import make_subplots
import torch
import torch.nn as nn
import plotly.graph_objects as go

from typing import Literal, Optional
from collections.abc import Callable

In [ ]:
%load_ext autoreload
%autoreload 2

from slrp_ev_data.read_old_slrpev_data import read_old_slrpev_data
from slrp_ev_data.read_new_slrpev_data import read_new_slrpev_data
from slrp_ev_data.read_ucsd_data import read_ucsd_data
from slrp_ev_data.train_test_split import train_test_split
from slrp_ev_data.window_generator import WindowGenerator

from slrp_ev_ts_forecasting.pacf import (
    get_pacf_values, get_threshold,
    sort_pacf_values,
    plot_df_pacf,
    get_pacf_values_complete_interval
)
from slrp_ev_data.feature_engineering import (
    feature_engineering,
    reverse_feature_engineering
                                              )
from slrp_ev_data.data_utils import get_data_frequency
from slrp_ev_data.normalization_and_standardization import (
    get_train_mean_and_std, get_train_min_and_max,
    get_rolling_scaling_column,
    apply_rolling_scaling,
    reverse_rolling_scaling
)

# Read different datasets

## Read old UCB data

In [ ]:
df_old = read_old_slrpev_data()
df_old

## Read new UCB data

In [ ]:
df_new = read_new_slrpev_data()

df_new

In [ ]:
# show gaps in the data
df_new["date"].diff().value_counts().iloc[1:]

## Read UCSD data

In [ ]:
df_ucsd = read_ucsd_data()
df_ucsd

In [ ]:
# show gaps in the data

df_ucsd["date"].diff().value_counts().iloc[1:]

## Visualization old vs new

### Differentiation Function

In [ ]:
lookback_15min_steps = 4
lookahead_15min_steps = 2
simple_test = pd.DataFrame(
    {
        "power": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "date": pd.date_range(start="2021-01-01", periods=11, freq="15min"),
    }
)
simple_test = get_rolling_scaling_column(
    simple_test,
    "rolling_standardize",
    lookahead_15min_steps,
    lookback_15min_steps,
    validate=False,
)
simple_test

A few tests

In [ ]:
lookback_15min_steps = 4
lookahead_15min_steps = 2
simple_test = pd.DataFrame(
    {
        "power": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "date": pd.date_range(start="2021-01-01", periods=11, freq="15min"),
    }
)
simple_test = get_rolling_scaling_column(
    simple_test, lookahead_15min_steps, lookback_15min_steps, validate=False
)

desired_result = pd.Series(
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 1.5, 2.5, 3.5, 4.5, 5.5],
    index=simple_test.index,
)
assert (simple_test["mean_power_for_diff"].dropna() == desired_result.dropna()).all()

lookback_15min_steps = 2
lookahead_15min_steps = 1
simple_test = pd.DataFrame(
    {
        "power": [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
        "date": pd.date_range(start="2021-01-01", periods=11, freq="5min"),
    }
)
simple_test = get_rolling_scaling_column(
    simple_test, lookahead_15min_steps, lookback_15min_steps, validate=False
)
desired_result = pd.Series(
    [np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, np.nan, 2.5, 3],
    index=simple_test.index,
)
assert (simple_test["mean_power_for_diff"].dropna() == desired_result.dropna()).all()

In [ ]:
dataset = df_new
cut_date = "2021-01-01"
scaling_param = get_rolling_scaling_column(
    dataset, "rolling_normalize", 96, 96 * 30, validate=False
)[("max_power_for_diff", "power")].loc[cut_date:]

fig = go.Figure()
fig.add_trace(
    go.Scatter(
        x=dataset[dataset["date"] >= cut_date]["date"],
        y=dataset[dataset["date"] >= cut_date]["power"],
        mode="lines",
        name="power",
        line=dict(color="red"),
    )
)
fig.add_trace(
    go.Scatter(
        x=scaling_param.index,
        y=scaling_param,
        mode="lines",
        name="Rolling Max Power",
        line=dict(color="blue"),
    )
)


fig.update_layout(
    title="Rolling Max Power vs Real power",
    xaxis_title="date",
    yaxis_title="Power (W)",
)
fig.show()

In [ ]:
scaling_mode = "rolling_standardize"

reverse_rolling_scaling(
    apply_rolling_scaling(
        df_new,
        get_rolling_scaling_column(df_new, scaling_mode, 96, 96 * 7, validate=False),
        scaling_mode,
    ),
    get_rolling_scaling_column(df_new, scaling_mode, 96, 96 * 7, validate=False),
    scaling_mode=scaling_mode,
)["power"].loc["2024-01-01":].plot()

### Final visualization

In [ ]:
start_date = pd.Timestamp("2021-01-25")
end_date = pd.Timestamp("2024-09-30")

number_of_time_ticks = 12
include_ucsd = True

lookahead_15min_steps = 96
lookback_15min_steps = 96 * 30
scale_power = True
scaling_mode: Literal["rolling_standardize", "rolling_normalize"] = "rolling_normalize"
log_transform = False

In [ ]:
# df_old_copy = df_old.copy()
df_new_copy = df_new.copy()
df_ucsd_copy = df_ucsd.copy()

if log_transform:
    CONSTANT = 0
    # df_old_copy["power"] = np.arcsinh(df_old_copy["power"] + CONSTANT)
    df_new_copy["power"] = np.arcsinh(df_new_copy["power"] + CONSTANT)
    if include_ucsd:
        df_ucsd_copy["power"] = np.arcsinh(df_ucsd_copy["power"] + CONSTANT)

if scale_power:
    # df_old_copy = apply_rolling_scaling(
    #     df_old_copy,
    #     get_rolling_scaling_column(
    #         df_old_copy,
    #         scaling_mode,
    #         lookahead_15min_steps,
    #         lookback_15min_steps,
    #         validate=False,
    #     ),
    #     scaling_mode=scaling_mode,
    # )
    df_new_copy = apply_rolling_scaling(
        df_new_copy,
        get_rolling_scaling_column(
            df_new_copy, scaling_mode, lookahead_15min_steps, lookback_15min_steps
        ),
        scaling_mode=scaling_mode,
    )
    if include_ucsd:
        df_ucsd_copy = apply_rolling_scaling(
            df_ucsd_copy,
            get_rolling_scaling_column(
                df_ucsd_copy, scaling_mode, lookahead_15min_steps, lookback_15min_steps
            ),
            scaling_mode=scaling_mode,
        )

# create the subframe
# sub_power_df_old = df_old_copy.query(
#     f"date >= '{start_date}' and date <= '{end_date}'"
# ).copy()
sub_power_df_new = df_new_copy.query(
    f"date >= '{start_date}' and date <= '{end_date}'"
).copy()

# Create the figure
fig = make_subplots(specs=[[{"secondary_y": False}]])

# Add the traces
# fig.add_trace(
#     go.Scatter(
#         x=sub_power_df_old["date"],
#         y=sub_power_df_old["power"],
#         name="Old data built from session info",
#     ),
#     secondary_y=False,
# )

# in order to show the missing data as missing, we resample
sub_power_df_new = (
    sub_power_df_new.set_index("date").resample("15min").mean().reset_index()
)
fig.add_trace(
    go.Scatter(
        x=sub_power_df_new["date"],
        y=sub_power_df_new["power"],
        name="New data built from power readings",
    ),
    secondary_y=False,
)

if include_ucsd:
    sub_power_df_ucsd = df_ucsd_copy.query(
        f"date >= '{start_date}' and date <= '{end_date}'"
    ).copy()
    sub_power_df_ucsd = (
        sub_power_df_ucsd.set_index("date").resample("15min").mean().reset_index()
    )
    fig.add_trace(
        go.Scatter(
            x=sub_power_df_ucsd["date"],
            y=sub_power_df_ucsd["power"],
            name="UCSD data",
        ),
        secondary_y=False,
    )

# Set the layout
fig.update_layout(
    title="Power Readings Over Time",
    xaxis=dict(title="Timestamp"),
    yaxis=dict(title="Power (W)", range=[0, 65000]),
    legend=dict(orientation="h", y=1.1, x=0.5),
)

# Set the x-axis ticks
desired_frequencies = [2, 4, 6, 12, 24, 48, 96, 168, 336, 720, 1440]
x_axis_freq = int(
    (end_date - start_date).total_seconds() / 3600 // number_of_time_ticks
)
x_axis_freq = max([freq for freq in desired_frequencies if freq <= x_axis_freq * 1.3])
x_axis_freq = f"{x_axis_freq}h"
fig.update_xaxes(
    tickmode="array",
    tickvals=pd.date_range(start_date, end_date, freq=x_axis_freq),
    ticktext=pd.date_range(start_date, end_date, freq=x_axis_freq).strftime(
        "%y-%m-%d %Hh"
    ),
    tickangle=75,
)

# Show the plot
fig.show()

Data on the morning of Sept. 26th 2023 was missing and interpolated

# Data Analysis

## Fourrier analysis for best time frequencies

Fourrier analysis can be used to identify the frequencies of the data, i.e. the most important time features.
See reference [here](https://www.tensorflow.org/tutorials/structured_data/time_series#time)

In [ ]:
import tensorflow as tf
import numpy as np

In [ ]:
fft = tf.signal.rfft(df_new["totalPower"].fillna(0))
f_per_dataset = np.arange(0, len(fft))

n_samples_5min = len(df_new["totalPower"])
intervals_5min_per_hour = 60 // 5
hours_per_year = 24 * 365.2524
years_per_dataset = n_samples_5min / (hours_per_year * intervals_5min_per_hour)

f_per_year = f_per_dataset / years_per_dataset
fig, ax = plt.subplots()
ax.step(f_per_year, np.abs(fft))
ax.set_xscale("log")
ax.set_ylim([0, max(plt.ylim())])
ax.set_xlim([0.1, max(plt.xlim())])
ax.set_xticks(
    [
        1,
        365.2524 / 7,
        365.2524 / 3.5,
        365.2524,
        365.2524 * 2,
        hours_per_year,
    ],
    labels=["1/Year", "1/week", "1/3.5 days", "1/day", "1/12hours", "1/hour"],
    rotation=45,
)
ax.tick_params(axis="x", colors="green", width=1, length=7)
_ = ax.set_xlabel("Frequency (log scale)")
plt.show()

As shown on the figure above, the most important frequencies are the week and the day (2 highest peaks). This means that important features are the time of the week (day of the week) and the time od the day (hour). Features like the time of the year (week number) or the time of the hour do not seem that important.

In addition, the high values for the very low frequencies suggest that we have a trend in the data

### Fourrier on old data

In [ ]:
fft = tf.signal.rfft(df_old["power"].fillna(0))
f_per_dataset = np.arange(0, len(fft))

n_samples_15min = len(df_old["power"])
intervals_15min_per_hour = 60 // 15
hours_per_year = 24 * 365.2524
years_per_dataset = n_samples_15min / (hours_per_year * intervals_15min_per_hour)

f_per_year = f_per_dataset / years_per_dataset
fig, ax = plt.subplots()
ax.step(f_per_year, np.abs(fft))
ax.set_xscale("log")
ax.set_ylim([0, max(plt.ylim())])
ax.set_xlim([0.1, max(plt.xlim())])
ax.set_xticks(
    [
        1,
        365.2524 / 7,
        365.2524 / 3.5,
        365.2524 / 1.5,
        365.2524,
        365.2524 * 2,
        hours_per_year,
    ],
    labels=[
        "1/Year",
        "1/week",
        "1/3.5 days",
        "1/1.5 days",
        "1/day",
        "1/12hours",
        "1/hour",
    ],
    rotation=45,
)
ax.tick_params(axis="x", colors="green", width=1, length=7)
_ = ax.set_xlabel("Frequency (log scale)")
plt.show()

### Fourrier on ucsd data

In [ ]:
fft = tf.signal.rfft(df_ucsd["power"].fillna(0))
f_per_dataset = np.arange(0, len(fft))

n_samples_15min = len(df_ucsd["power"])
intervals_15min_per_hour = 60 // 15
hours_per_year = 24 * 365.2524
years_per_dataset = n_samples_15min / (hours_per_year * intervals_15min_per_hour)

f_per_year = f_per_dataset / years_per_dataset
fig, ax = plt.subplots()
ax.step(f_per_year, np.abs(fft))
ax.set_xscale("log")
ax.set_ylim([0, max(plt.ylim())])
ax.set_xlim([0.1, max(plt.xlim())])
ax.set_xticks(
    [
        1,
        365.2524 / 7,
        365.2524 / 3.5,
        365.2524 / 1.5,
        365.2524,
        365.2524 * 2,
        hours_per_year,
    ],
    labels=[
        "1/Year",
        "1/week",
        "1/3.5 days",
        "1/1.5 days",
        "1/day",
        "1/12hours",
        "1/hour",
    ],
    rotation=45,
)
ax.tick_params(axis="x", colors="green", width=1, length=7)
_ = ax.set_xlabel("Frequency (log scale)")
plt.show()

## Analyze lags correlation

In [ ]:
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.stattools import pacf
from darts.utils.statistics import plot_acf, plot_pacf

The following test ensures that the data is stationary. Stationary is *required* because it is an assumption of the statistical models. To make data stationary, we can use differentiation.

In [ ]:
def check_stationarity(series):
    # Copied from https://machinelearningmastery.com/time-series-data-stationary-python/

    result = adfuller(series.values)

    print("ADF Statistic: %f" % result[0])
    print("p-value: %f" % result[1])
    print("Critical Values:")
    for key, value in result[4].items():
        print("\t%s: %.3f" % (key, value))

    if (result[1] <= 0.05) & (result[4]["5%"] > result[0]):
        print("\u001b[32mStationary\u001b[0m")
    else:
        print("\x1b[31mNon-stationary\x1b[0m")


print("Checking stationarity of ucsd data:")
check_stationarity(df_ucsd["power"])
print("Checking stationarity of old data:")
check_stationarity(df_old["power"])
print("Checking stationarity of new data:")
check_stationarity(df_new["totalPower"].fillna(0))

The blue area in the ACF and PACF plots, which depicts the 95% confidence interval and is in indicator for the significance threshold. That means, anything within the blue area is statistically close to zero and anything outside the blue area is statistically non-zero.

- If there are several autocorrelation lags that are significantly non-zero (ACF), the time series is non-random.
- The significant lags (outside blue area) of the PACF mean that we data is autoregressive (could be a linear combination of all those lags)
- The significant lags (outside blue area) of the ACF mean that we can build a moving average model

### Analyze pacf if missing data

In [ ]:
def get_df_complete_intervals(
    df: pd.DataFrame,
    date_column: str,
    frequency: Literal["5Min", "15Min"],
) -> pd.DataFrame:
    """Returns the complete intervals. there is data for the specified start_complete \
        and end_complete dates (so use >= and <=)"""
    df_complete_intervals = df.copy()
    # find the difference between two data points
    df_complete_intervals["diff"] = df_complete_intervals[date_column].diff()

    # dates are missing if the difference is not the frequency
    df_complete_intervals = df_complete_intervals[
        df_complete_intervals["diff"] > pd.Timedelta(frequency)
    ]
    if df_complete_intervals.empty:
        # Then we have no missing interval!
        return pd.DataFrame(
            {
                "start_complete": (df[date_column].min()),
                "end_complete": df[date_column].max(),
            },
            index=[0],
        )

    # define start and end dates of missing intervals
    for i in range(df_complete_intervals.shape[0]):
        index = df_complete_intervals.iloc[i].name
        df_complete_intervals.loc[index, "end_complete"] = (
            df_complete_intervals.loc[index, date_column]
            - df_complete_intervals.loc[index, "diff"]
        )
        if i == 0:
            df_complete_intervals.loc[index, "start_complete"] = df.iloc[0][date_column]
        else:
            df_complete_intervals.loc[index, "start_complete"] = (
                df_complete_intervals.iloc[i - 1][date_column]
            )

    # add the last missing interval
    last_row = pd.DataFrame(
        {
            "start_complete": (df_complete_intervals.iloc[-1][date_column]),
            "end_complete": df.iloc[-1][date_column],
        },
        index=[df_complete_intervals.index[-1] + 1],
    )
    df_complete_intervals = pd.concat([df_complete_intervals, last_row], axis=0)

    return df_complete_intervals

In [ ]:
data = df_new

df_complete_intervals = get_df_complete_intervals(data, "date", "15Min")
df_complete_intervals

In [ ]:
nb_of_days_for_pacf = 70

colors = sns.color_palette("tab10")
dict_colors = {}
dict_of_pacfs = {}
total_number_of_intervals = 0
minimum_number_intervals_covered_if_using_pacf = 0
average_pacf = pd.DataFrame()
for complete_interval in df_complete_intervals.itertuples():
    filtered_df = data.query(
        f"date >= '{complete_interval.start_complete}' and date <= '{complete_interval.end_complete}'"
    )
    length_data = filtered_df.shape[0]
    total_number_of_intervals += length_data
    print(f"Len of filtered_df: {filtered_df.shape[0]}")
    if length_data < nb_of_days_for_pacf * 24 * 4 * 2:
        print("Skipping interval because not enough data points")
        continue
    minimum_number_intervals_covered_if_using_pacf += (
        length_data - nb_of_days_for_pacf * 24 * 4
    )

    pacf_df, interval = get_pacf_values_complete_interval(
        1,
        filtered_df,
        nb_of_days_for_pacf=nb_of_days_for_pacf,
        nb_of_steps_to_predict=1,
        return_confidence_interval=True,
    )
    print(interval)
    average_pacf = pd.concat([average_pacf, pacf_df], axis=1)
    interval_str_name = f"{complete_interval.start_complete.date()}_{complete_interval.end_complete.date()}"
    dict_of_pacfs[interval_str_name] = pacf_df
    dict_colors[interval_str_name] = colors[len(dict_colors)]

average_pacf = pd.DataFrame({"PACF": average_pacf.mean(axis=1)})
print(
    f"With a pacf of {nb_of_days_for_pacf} days, we cover at least {minimum_number_intervals_covered_if_using_pacf/total_number_of_intervals*100:.2f}% of the intervals"
)

Below, we plot the pacf of the complete intervals of the data. Good news, they look very similar. Therefore, we can probably take their average and assume that it's what the overall pacf looks like

In [ ]:
def add_threshold(fig, df, color_name):
    threshold, index_of_farther_point = get_threshold(df, verbose=False)
    print(f"Farthest point for {color_name} is at index {index_of_farther_point}")
    # Add a horizontal line at y=0
    fig.add_shape(
        type="line",
        x0=0,
        y0=threshold,
        x1=len(df) - 1,
        y1=threshold,
        line=dict(color=f"rgb{dict_colors[interval_str_name]}", width=2),
    )


# Create a bar plot using Plotly
fig = go.Figure()

for interval_str_name, pacf_df in dict_of_pacfs.items():
    fig.add_trace(
        go.Bar(
            x=pacf_df.index,
            y=pacf_df["PACF"],
            name=interval_str_name,
            marker_color=f"rgb{dict_colors[interval_str_name]}",
        )
    )

    # add threshold lines
    add_threshold(fig, pacf_df, interval_str_name)

# Customize the x-axis ticks and labels
fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=[96, 96 * 7],
        ticktext=["1 day", "1 week"],
    ),
    title="Partial Auto-Correlation Function (PACF)",
    xaxis_title="Lag",
    yaxis_title="PACF",
    plot_bgcolor="white",  # Set the plot background color to white
)

# Show the plot
fig.show()

### Pacf with dart package

In [ ]:
series = TimeSeries.from_dataframe(
    df_ucsd[df_ucsd["date"] >= "2024"],
    time_col="date",
    value_cols="power",
    fill_missing_dates=True,
    freq="15Min",
)


fig, ax = plt.subplots()

downsample_hours = 2

corr_params = {
    "ts": series.resample(f"{downsample_hours}h"),
    "m": 4,
    "alpha": 0.05,
    "max_lag": int(96 / (4 * downsample_hours) * 32),
}

ticks_params = {
    "ticks": [
        96 / (4 * downsample_hours),
        96 / (4 * downsample_hours) * 7,
        96 / (4 * downsample_hours) * 14,
        96 / (4 * downsample_hours) * 21,
        96 / (4 * downsample_hours) * 30,
    ],
    "labels": ["1day", "1 week", "2 weeks", "3 weeks", "month"],
}

plot_acf(axis=ax, **corr_params)

ax.set_title("Auto-correlation-function of old data for the lags of a month")


ax.tick_params(axis="x", colors="green", width=1, length=7)
ax.set_xticks(**ticks_params)
ax.tick_params(axis="x", colors="green", width=1, length=7)
ax.set_xlabel("Lag")
plt.show()

fig, ax = plt.subplots()
plot_pacf(axis=ax, **corr_params)
ax.set_title("Partial auto-correlation-function of old data for the lags of a month")

ax.tick_params(axis="x", colors="green", width=1, length=7)
ax.set_xticks(**ticks_params)
ax.tick_params(axis="x", colors="green", width=1, length=7)
ax.set_xlabel("Lag")
ax.set_ylim(-0.3, 0.3)
plt.show()

### Analyze Differences in downsampling value

In [ ]:
pacf_df_2h = get_pacf_values(2, df_old, nb_of_days_for_pacf=50)
pacf_df_1h = get_pacf_values(1, df_old, nb_of_days_for_pacf=30)
pacf_df_15min = get_pacf_values(1 / 4, df_old, nb_of_days_for_pacf=20)

In [ ]:
COLORS = {
    "PACF 2h": "blue",
    "PACF 1h": "green",
    "PACF 15min": "red",
}


def add_threshold(fig, df, color_name):
    threshold, index_of_farther_point = get_threshold(df, verbose=False)
    print(f"Farthest point for {color_name} is at index {index_of_farther_point}")
    # Add a horizontal line at y=0
    fig.add_shape(
        type="line",
        x0=0,
        y0=threshold,
        x1=len(df) - 1,
        y1=threshold,
        line=dict(color=COLORS[color_name], width=2),
    )


# Create a bar plot using Plotly
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=pacf_df_2h.index,
        y=pacf_df_2h["PACF"],
        name="PACF 2h",
        marker_color=COLORS["PACF 2h"],
    )
)
fig.add_trace(
    go.Bar(
        x=pacf_df_1h.index,
        y=pacf_df_1h["PACF"],
        name="PACF 1h",
        marker_color=COLORS["PACF 1h"],
    )
)
fig.add_trace(
    go.Bar(
        x=pacf_df_15min.index,
        y=pacf_df_15min["PACF"],
        name="PACF 15min",
        marker_color=COLORS["PACF 15min"],
    )
)

# add threshold lines
add_threshold(fig, pacf_df_2h, "PACF 2h")
add_threshold(fig, pacf_df_1h, "PACF 1h")
add_threshold(fig, pacf_df_15min, "PACF 15min")

# Customize the x-axis ticks and labels
fig.update_layout(
    xaxis=dict(
        tickmode="array",
        tickvals=[96, 96 * 7],
        ticktext=["1 day", "1 week"],
    ),
    title="Partial Auto-Correlation Function (PACF)",
    xaxis_title="Lag",
    yaxis_title="PACF",
    plot_bgcolor="white",  # Set the plot background color to white
)

# Show the plot
fig.show()

### Compare long and short optimized lags methods

In [ ]:
def plot_selected_lags(df, downsample_hours: float, number_of_lags_to_keep: int = 96):
    # Create a subplot layout with 2 rows and 1 column
    fig = make_subplots(
        rows=4,
        cols=1,
        row_heights=[
            0.5,
            0.25,
            0.5,
            0.25,
        ],  # Specify the relative heights of each subplot
        shared_xaxes=True,  # Share the x-axis between subplots
        vertical_spacing=0.1,  # Space between subplots
    )

    for i, (name, nb_of_steps_to_predict) in enumerate([("short", 96), ("long", 1)]):
        pacf_df, interval = get_pacf_values(
            downsample_hours,
            df,
            nb_of_days_for_pacf=50,
            nb_of_steps_to_predict=nb_of_steps_to_predict,
            return_confidence_interval=True,
        )
        # Add the first figure (PACF bar plot) to the first row
        plot_df_pacf(
            pacf_df,
            number_of_lags_to_keep,
            interval=interval,
            figure=fig,
            figure_number=i,
        )

    # Update layout
    fig.update_layout(
        title="PACF and Selected Lags<br>The short method is at the top, the long method is at the bottom.",
        height=600,  # Total height of the figure
    )
    fig.update_yaxes(title_text="PACF", row=1, col=1)
    fig.update_xaxes(title_text="Lag", row=4, col=1)

    # Show the plot
    fig.show()


plot_selected_lags(df_old, 1, 96 * 2)

### Test pacf.py functions

In [ ]:
pacf_df = get_pacf_values(
    1,
    df_old,
    nb_of_days_for_pacf=30,
    nb_of_steps_to_predict=2,
)

In [ ]:
plot_df_pacf(pacf_df, 96 * 2)

## Distribution of monthly peak power

In [ ]:
# show max power for each month for each year
# June 2024 is omitted because not complete
monthly_max_power = (
    df_new[df_new["recordTimestamp"] < "2024-06-01"]
    .groupby([df_new["recordTimestamp"].dt.year, df_new["recordTimestamp"].dt.month])[
        "totalPower"
    ]
    .max()
)

# plot distribution of max monthly power, with a different color for each year
fig, ax = plt.subplots(figsize=(10, 5))
monthly_max_power.unstack().plot(kind="bar", ax=ax)
ax.set_ylabel("Max Power (W)")
ax.set_xlabel("Month")
ax.set_title("Max Power per Month (June 2024 omitted because not complete)")
plt.grid(True)
plt.show()

### Distribution of when the peaks occur (hour)

In [ ]:
choice_mode_distribution: Literal["peak_power", "peak_hour", "peak_day_of_week"] = (
    "peak_power"
)
frequency_distribution: Literal["each_month", "each_year"] = "each_year"

In [ ]:
# get power grouped for each month for each year
# June 2024 is omitted because not complete
mask = df_new["recordTimestamp"] < "2024-06-01"
if frequency_distribution == "each_month":
    mask = mask & (df_new["recordTimestamp"].dt.year > 2020)
df_for_each_year_month = df_new[mask].groupby(
    [df_new["recordTimestamp"].dt.year, df_new["recordTimestamp"].dt.month]
)["totalPower"]
if choice_mode_distribution == "peak_power":
    df_for_each_year_month = df_for_each_year_month.max()
elif choice_mode_distribution == "peak_day_of_week":
    df_for_each_year_month = df_for_each_year_month.idxmax().apply(
        lambda x: df_new.loc[x, "recordTimestamp"].day_of_week
    )
elif choice_mode_distribution == "peak_hour":
    df_for_each_year_month = df_for_each_year_month.idxmax().apply(
        lambda x: df_new.loc[x, "recordTimestamp"].hour
    )


fig, ax = plt.subplots(figsize=(10, 5))

if choice_mode_distribution == "peak_day_of_week":
    max_x_axis = 7
    bin_size = 1
elif choice_mode_distribution == "peak_hour":
    max_x_axis = 23
    bin_size = 1
elif choice_mode_distribution == "peak_power":
    max_x_axis = 60000
    bin_size = 2500
# Define the bin edges
bin_edges = np.arange(0, max_x_axis, bin_size) - bin_size / 2
x = np.linspace(0, max_x_axis, 1000)  # Define the x range for the PDF

df_for_each_year_month = df_for_each_year_month.unstack()
if frequency_distribution == "each_month":
    df_for_each_year_month = df_for_each_year_month.transpose()


colors = plt.cm.nipy_spectral(
    np.linspace(0, 1, len(df_for_each_year_month.index))
)  # Generate a color for each year

ax.hist(
    [df_for_each_year_month.loc[year] for year in df_for_each_year_month.index],
    bins=bin_edges,  # Use the defined bin edges
    color=colors,
    alpha=1,
    stacked=True,
    density=True,  # Normalize the histogram
    align="mid",  # Center the bars on the ticks
)

y_max_list = []
for color, year in zip(colors, df_for_each_year_month.index):
    # here "year" can refer to year or month, depending on frequence_distribution
    data = df_for_each_year_month.loc[year].dropna()

    # ax.hist(
    #     df_for_each_year_month.loc[year],
    #     bins=bin_edges,  # Use the defined bin edges
    #     color=color,
    #     alpha=0.5,
    #     label=year,
    #     density=True,  # Normalize the histogram
    # )

    # Fit a normal distribution to the data
    mu, std = norm.fit(data)

    # Plot the PDF
    std = max(std, 0.01)  # 0.01 is a small number to avoid division by zero
    p = norm.pdf(x, mu, std)
    y_max_list.append(max(p))
    ax.plot(x, p, linewidth=2, color=color, label=f"Gaussian {year}")

ax.set_ylim(
    0, np.median(y_max_list) * 2
)  # set y limit to 1.5 times the median of the maximum y value of the gaussian curves, so that the gaussian curves are all visible
if choice_mode_distribution == "peak_day_of_week":
    ax.set_xlabel("Day of the week of the peak")
elif choice_mode_distribution == "peak_hour":
    ax.set_xlabel("Hour of the peak")
elif choice_mode_distribution == "peak_power":
    ax.set_xlabel("Peak Power (Watts)")

ax.set_ylabel("Density")
if frequency_distribution == "each_year":
    ax.legend(title="Year")
elif frequency_distribution == "each_month":
    ax.legend(title="Month")
plt.title(
    f"Histogram of monthly {choice_mode_distribution.replace("_", " ")} with Gaussian Distribution"
)
plt.show()

## Load Curve

In [ ]:
# Parameters
split_by_year: bool = True
data = df_new

In [ ]:
def get_load_duration_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    # Define the bin edges
    bins = np.linspace(
        df["power"].min() - 0.001, df["power"].max(), 1000
    )  # Adjust the number of bins as needed

    # Use pandas.cut to bin the data into intervals
    df["bin"] = pd.cut(df["power"], bins=bins)

    # Use value_counts to count the frequency of each bin
    histogram = df["bin"].value_counts().sort_index()

    # Convert the histogram to a DataFrame
    histogram = pd.DataFrame({"bin": histogram.index, "frequency": histogram.values})
    histogram["bin"] = histogram["bin"].apply(lambda x: round(x.left))

    histogram = histogram.sort_values(by="bin", ascending=False)

    histogram["cumulative_frequency"] = (
        histogram["frequency"].cumsum() / histogram["frequency"].sum()
    )

    return histogram


fig, ax = plt.subplots()

colors = plt.cm.viridis(np.linspace(0, 1, len(data["date"].dt.year.unique())))
if split_by_year:
    for group_iter, color in zip(data.groupby(data["date"].dt.year), colors):
        name, group = group_iter
        histogram = get_load_duration_df(group)
        ax.plot(
            histogram["cumulative_frequency"],
            histogram["bin"],
            label=name,
            color=color,
        )
        ax.legend(title="Year")
else:
    histogram = get_load_duration_df(data)
    ax.plot(histogram["cumulative_frequency"], histogram["bin"])


# plot a dash line at the power value of x chargers
total_number_of_chargers_to_show = 8
for charger_number in range(1, total_number_of_chargers_to_show + 1):
    ax.axhline(
        y=charger_number * 6600,
        color="gray",
        alpha=0.5,
        linestyle="--",
        zorder=-1,  # This ensures the line is plotted behind existing elements
    )
ax.set_xlabel("Duration (%)")
ax.set_ylabel("Power (Watts)")
ax.set_title(
    f"Load Duration Curve\nThe dashed lines represent the steps from 0 to {total_number_of_chargers_to_show} chargers."
)

ax.grid(True, axis="x")

## Heatmap of peak distribution

In [ ]:
power_mode: Literal["max", "mean"] = "max"
frequency_mode: Literal["hour_of_day", "day_of_week"] = "day_of_week"
color_code = "YlOrRd"

In [ ]:
# we want a heatmap that shows the power consumption for each hour of the day for each month of the year

# June 2024 is omitted because not complete
mask = (df_new["date"] < "2024-06-01") & (df_new["date"].dt.year > 2020)

if frequency_mode == "hour_of_day":
    df_for_each_month_and_hour = df_new[mask].groupby(
        [df_new["date"].dt.month, df_new["date"].dt.hour]
    )["power"]
elif frequency_mode == "day_of_week":
    df_for_each_month_and_hour = df_new[mask].groupby(
        [df_new["date"].dt.month, df_new["date"].dt.day_of_week]
    )["power"]

if power_mode == "max":
    df_for_each_month_and_hour = df_for_each_month_and_hour.max()
elif power_mode == "mean":
    df_for_each_month_and_hour = df_for_each_month_and_hour.mean()

df_for_each_month_and_hour = df_for_each_month_and_hour.unstack()

df_for_each_month_and_hour.index.rename("Month", inplace=True)
# write month as a string
df_for_each_month_and_hour.index = df_for_each_month_and_hour.index.map(
    lambda x: pd.Timestamp(year=2024, month=x, day=1).strftime("%B")
)

# plot the heatmap
fig, ax = plt.subplots(figsize=(10, 5))

# Create the heatmap
sns.heatmap(
    df_for_each_month_and_hour,
    cmap=color_code,
    cbar_kws={"label": "Power (W)"},
    ax=ax,
)

# Remove gridlines
ax.grid(False)

# Add titles and labels
plt.title(
    f"{power_mode.capitalize()} Power Consumption for Each {frequency_mode.replace('_', ' ').capitalize()} for Each Month (2020 omitted)"
)
plt.xlabel(f"{frequency_mode.replace('_', ' ').capitalize()}")
plt.ylabel("Month")

# Display the plot
plt.show()

In [ ]:
df_test = df_new[mask].groupby(
    [
        df_new["date"].dt.month,
        df_new["date"].dt.hour,
        df_new["date"].dt.day_of_week,
    ]
)["power"]

if power_mode == "max":
    df_test = df_test.max()
elif power_mode == "mean":
    df_test = df_test.mean()

df_test = df_test.unstack().unstack()

# write month as a string
df_test.index = df_test.index.map(
    lambda x: pd.Timestamp(year=2024, month=x, day=1).strftime("%B")
)

df_all = pd.DataFrame()
for i in range(7):
    df_day_i = df_test.loc[:, i]
    df_day_i.columns = pd.MultiIndex.from_tuples(
        [
            (pd.Timestamp(year=2024, month=12, day=i + 2).strftime("%A"), col)
            for col in df_day_i.columns
        ]
    )
    df_all = pd.concat([df_all, df_day_i], axis=1)

# plot the heatmap
fig, ax = plt.subplots(figsize=(18, 5))

# Create the heatmap
sns.heatmap(
    df_all,
    cmap=color_code,
    cbar_kws={"label": "Power (W)"},
    ax=ax,
)

# Remove gridlines
ax.grid(False)

# Set the number of ticks
label_indices = np.arange(0, len(df_all.columns), 3)
ax.set_xticks(label_indices)

labels = df_all.columns[label_indices]
# Customize the tick labels to show two levels
ax.set_xticklabels(
    [f"{hour}\n{day}" if hour == 12 else f"{hour}" for day, hour in labels], rotation=0
)

# Add titles and labels
# Add titles and labels
plt.title(
    f"{power_mode.capitalize()} Power Consumption for each hour of each day of each month (2020 omitted)"
)
plt.xlabel("Day and Hour")
plt.ylabel("Month")

# Display the plot
plt.show()

## Trend analysis

We are going to analyze the evolution of the mean daily (or weekly values) values for the ucsd data. We can see that the data is not stationary. Persistence from the previous timestep seems to be the best predictor (highest R2 score). Therefore, to make the data stationary, we should subtract to each day/week the average of the previous day/week.

In [ ]:
from scipy.optimize import curve_fit
from sklearn.metrics import r2_score


# Define the polynomial function
def polynomial_2(x, a, b, c):
    return a * x**2 + b * x + c


def exponential(x, a, b, c):
    return np.exp(x * a) + c


def pow(x, a, b, c):
    return a * x**b + c


def log(x, a, b, c):
    return a * np.log(x * b) + c


def fit_curve(x, y, function) -> tuple:
    # Fit the curve
    params, _ = curve_fit(function, x, y, maxfev=5000)
    x_fit = np.linspace(x.min(), x.max(), max(100, x.shape[0]))
    y_fit = function(x_fit, *params)

    r2 = r2_score(y, y_fit)
    return x_fit, y_fit, r2


def plot_mean_values_evolution(data: pd.DataFrame, frequency_for_mean: str = "D"):
    data = data.copy()
    data = data.set_index("date").resample(frequency_for_mean).mean().reset_index()
    data = data.rename(columns={"power": "energy"})
    data["energy"] = (
        data["energy"] / 4
    )  # WARNING: This is not true for new slrp-ev data since
    # the power frequency is 5 min and not 15 min

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=data["date"], y=data["energy"], name="Real Data"))

    # Add poly 2 curve
    data_for_fit = data.dropna()[1:]  # 1: to remve 0 from the index
    x_fit, y_fit, r2 = fit_curve(
        data_for_fit.index, data_for_fit["energy"], polynomial_2
    )
    fig.add_trace(
        go.Scatter(
            x=data["date"].iloc[x_fit], y=y_fit, name=f"Polynomial Fit - R2 {r2:.3f}"
        )
    )

    # Add pow curve
    x_fit, y_fit, r2 = fit_curve(data_for_fit.index, data_for_fit["energy"], pow)
    fig.add_trace(
        go.Scatter(x=data["date"].iloc[x_fit], y=y_fit, name=f"Pow Fit - R2 {r2:.3f}")
    )

    # Add exp curve
    try:
        x_fit, y_fit, r2 = fit_curve(
            data_for_fit.index, data_for_fit["energy"], exponential
        )
        fig.add_trace(
            go.Scatter(
                x=data["date"].iloc[x_fit],
                y=y_fit,
                name=f"Exponential Fit - R2 {r2:.3f}",
            )
        )
    except RuntimeError:
        print("Not able to fit exp. Try reducing the frequency")

    # Add log curve
    try:
        x_fit, y_fit, r2 = fit_curve(data_for_fit.index, data_for_fit["energy"], log)
        fig.add_trace(
            go.Scatter(
                x=data["date"].iloc[x_fit], y=y_fit, name=f"Log Fit - R2 {r2:.3f}"
            )
        )
    except RuntimeError:
        print("Not able to fit log. Try reducing the frequency")

    # Add persistence
    data_persistence = data.copy().dropna()
    data_persistence["energy_persistence"] = data_persistence["energy"].shift(1)
    data_persistence = data_persistence.dropna()
    y_persistence = data_persistence["energy_persistence"]
    x_persistence = data_persistence["date"]
    y_real = data_persistence["energy"]
    fig.add_trace(
        go.Scatter(
            x=x_persistence,
            y=y_persistence,
            name=f"Persistence - R2 {r2_score(y_persistence, y_real):.3f}",
        )
    )

    # Customize the x-axis ticks and labels
    fig.update_layout(
        title=f"Trend evolution ({frequency_for_mean})",
        xaxis_title="Date",
        yaxis_title=f"Average {frequency_for_mean} energy",
        plot_bgcolor="white",  # Set the plot background color to white
    )

    # Show the plot
    fig.show()


plot_mean_values_evolution(df_ucsd, "W")

# Data windowing

In [ ]:
df_train, df_test = train_test_split(df_new, fraction_in_train=0.8)  # type: ignore

normalize_params = get_train_min_and_max(df_train)

In [ ]:
x_dim = 32
lookahead = 8
w1 = WindowGenerator(
    input_width=x_dim,
    label_width=x_dim,
    shift=lookahead,
    train_df=feature_engineering(df_train, normalize_parameters=normalize_params),
    test_df=feature_engineering(df_test, normalize_parameters=normalize_params),
    get_val_from_shuffled_train=False,
    label_columns=["power", "date"],
    overlapping_windows=False,
    batch_size=1000,
    verbose=True,
)

w1

In [ ]:
w1.train

In [ ]:
w1.plot_train_val_split_selection()

In [ ]:
pd.to_datetime(next(iter(w1.train.take(1).as_numpy_iterator()))[0][0][1:, 0], unit="s")

In [ ]:
item_number = 0
inputs_first_batch = next(iter(w1.train))[0]
start = pd.to_datetime(inputs_first_batch[item_number][0][0], unit="s")
end = pd.to_datetime(inputs_first_batch[item_number][-1][0], unit="s")

print(f"Start: {start}, end: {end} of item {item_number}")

In [ ]:
item_number = 1  # works if batch size is >=2
inputs_first_batch = next(iter(w1.train))[0]
start = pd.to_datetime(inputs_first_batch[item_number][0][0], unit="s")
end = pd.to_datetime(inputs_first_batch[item_number][-1][0], unit="s")

print(f"Start: {start}, end: {end} of item {item_number}")

In [ ]:
df_train.groupby(df_train["date"].dt.minute).count()

In [ ]:
df_train_eng = feature_engineering(df_train, normalize_parameters=normalize_params)

In [ ]:
reverse_feature_engineering(
    df_train_eng, normalize_parameters=normalize_params
).groupby(
    reverse_feature_engineering(df_train_eng, normalize_parameters=normalize_params)[
        "date"
    ].dt.minute
).count()

In [ ]:
reverse_feature_engineering(df_train_eng)

In [ ]:
flat_inputs, flat_labels = w1.flatten_dataset(
    w1.train, cols_keep_last_value=["date"], label_cols_to_flatten=["date"]
)

In [ ]:
flat_labels

In [ ]:
for batch in w1.train:
    inputs, labels = batch
    print(f"inputs.shape: {inputs.shape}, labels.shape: {labels.shape}")

In [ ]:
w1.column_indices

In [ ]:
dataset = w1.convert_to_torch_dataset(w1.train, ["workday", "time_window"], ["power"])

In [ ]:
inputs, outputs = dataset.get_full_data()

In [ ]:
flat_inputs, flat_labels = w1.flatten_dataset(
    w1.train,
    cols_keep_last_value=["workday", "time_window"],
    cols_keep_some_values=[
        {"col_name": "date", "indexes_to_keep": w1.input_width - np.array([1, 4])}
    ],
    label_cols_to_flatten=["date"],
)

In [ ]:
columns = ["time_window", "workday"]
one_hot_length = flat_inputs[columns].apply(lambda x: len(x.unique()))
# we do not want to encode boolean arrays (with only two values)
if one_hot_length[one_hot_length <= 2].shape[0] > 0:
    columns_to_ignore = one_hot_length[one_hot_length <= 2].index.tolist()
    print(
        f"WARNING: Column(s) {columns_to_ignore} are boolean (only two possible values) and will not be one-hot encoded"
    )
    columns = [col for col in columns if col not in columns_to_ignore]

In [ ]:
w1.plot()

In [ ]:
# convert the date to a timestamp
df_train_for_tf = df_train.copy()

df_train_for_tf["date"] = df_train_for_tf["date"].astype("int64") // 10**9

In [ ]:
{"a": 6} | {"b": 7}

# Tests

### Sktime tests

In [ ]:
from sktime.datasets import load_airline
from sktime.forecasting.fbprophet import Prophet
from sktime.utils.plotting import plot_series

In [ ]:
y = load_airline()

In [ ]:
# this is the data known in December 1957
y_train = y[:-36]

In [ ]:
forecaster = Prophet(
    freq="1M",
    seasonality_mode="multiplicative",
    # n_changepoints=int(len(y_train) / 12),
    # add_country_holidays={"country_name": "US"},
    yearly_seasonality=True,
    weekly_seasonality=False,
    daily_seasonality=False,
    # the three growth arguments go together.
    # Don't forget that the data is normalize (btw 0 and 1)
    # growth_floor=0,
    # growth_cap=1,
    # growth="logistic",
)

In [ ]:
forecaster.fit(y=y_train)

In [ ]:
lookahead = 12
y_pred = forecaster.predict(fh=np.arange(1, lookahead + 1))

In [ ]:
plot_series(
    y_train,
    y[-36 : -36 + lookahead],
    y_pred,
    labels=["y_train", "y_test", "y_pred"],
)

In [ ]:
y.loc[pd.timedelta_range(y.index[0], y.index[0] + pd.Timedelta(days=40))]

### Darts tests

In [ ]:
from darts.models import NaiveSeasonal
from darts.metrics import mse
from darts.models import (
    ExponentialSmoothing,
)  # very long... (didn't run a less than 12 hours)
from darts.models import NBEATSModel
from darts.dataprocessing.transformers import Scaler
from darts import concatenate
from darts.utils.timeseries_generation import datetime_attribute_timeseries as dt_attr
from darts.models import BlockRNNModel
from darts.models import TCNModel
from darts.models import TransformerModel
from darts.models import TFTModel
from darts.models import ARIMA
from darts.models import Prophet as ProphetDarts

In [ ]:
LOOKAHEAD = 16
X_DIM = 96

In [ ]:
series = TimeSeries.from_dataframe(df_old, time_col="date", value_cols="power")
train, val = series.split_before(pd.Timestamp("2023-09-01"))

fig, ax = plt.subplots()

train.plot(label="training", ax=ax)
val.plot(label="validation", ax=ax)

# Plot the month on a secondary axis
ax2 = ax.twinx()
dt_attr(series, "hour", dtype=np.float32, cyclic=True).plot(label="tom", ax=ax2)

ax.set_xlim(pd.Timestamp("2024-02-01"), pd.Timestamp("2024-03-01"))

# Set labels for the axes
ax.set_xlabel("Date")
ax.set_ylabel("Power")
ax2.set_ylabel("Month")

# Create a legend
lines, labels = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax2.legend(lines + lines2, labels + labels2, loc="upper left")

# Display the plot
plt.show()

In [ ]:
def get_covariates(ts: TimeSeries) -> TimeSeries:
    series_cov = concatenate(
        [
            dt_attr(ts, "day_of_week", dtype=np.float32, cyclic=True),
            dt_attr(ts, "day_of_year", dtype=np.float32, cyclic=True),
            dt_attr(ts, "hour", dtype=np.float32, cyclic=True),
        ],
        axis="component",
    )
    # default scaler is min-max scaler
    series_cov = Scaler().fit_transform(series_cov)
    return series_cov

In [ ]:
def eval_model(
    model, series: TimeSeries, mode: Literal["statistical", "deepl"]
) -> TimeSeries:
    # get train, validation and test sets
    fraction_in_train = 0.7
    fraction_in_val = 0.2
    data_length = len(series)
    split_train_index = int(data_length * fraction_in_train)
    train = series[: split_train_index - 1]
    split_val_index = int(data_length * (fraction_in_train + fraction_in_val))
    val = series[split_train_index : split_val_index - 1]
    test = series[split_val_index:]

    # scale data (min max scaling)
    scaler = Scaler()
    train_scaled = scaler.fit_transform(train)
    val_scaled = scaler.transform(val)
    test_scaled = scaler.transform(test)

    # covariates
    covariates = get_covariates(series)

    if mode == "statistical":
        model.fit(train_scaled)

    elif mode == "deepl":
        model.fit(
            train_scaled,
            past_covariates=covariates,
            val_series=val_scaled,
            val_past_covariates=covariates,
            epochs=3,
            verbose=True,
        )

    hfc_params = {
        "series": test_scaled,
        "forecast_horizon": LOOKAHEAD,
        "verbose": True,
    }
    if mode == "deepl":
        hfc_params["retrain"] = False
        hfc_params["past_covariates"] = covariates
    forecast = model.historical_forecasts(last_points_only=True, **hfc_params)
    forecast = scaler.inverse_transform(forecast)

    print(f"model {model} obtains RMSE: {np.sqrt(mse(test, forecast)):.2f}")

    return forecast


def plot_forecast(
    forecast: TimeSeries,
    series: TimeSeries,
    start_date=pd.Timestamp("2024-02-01"),
    end_date=pd.Timestamp("2024-03-01"),
):
    forecast = forecast.resample("15min")
    print(f"RMSE after resampling: {np.sqrt(mse(series, forecast)):.2f}")
    fig, ax = plt.subplots()
    series.plot(label="actual", ax=ax)
    forecast.plot(label="forecast", ax=ax)
    ax.set_xlim(start_date, end_date)

    plt.show()

In [ ]:
downsample_hours = 1
model = NaiveSeasonal(K=int(96 / (4 * downsample_hours)))


forecast = eval_model(
    model, series.resample(f"{downsample_hours}h"), mode="statistical"
)


plot_forecast(forecast, series)

In [ ]:
downsample_hours = 4
model = ARIMA(p=1, d=0, q=1, seasonal_order=(2, 1, 2, int(96 / (4 * downsample_hours))))
forecast = eval_model(
    model, series.resample(freq=f"{downsample_hours}h"), mode="statistical"
)
plot_forecast(forecast, series)

In [ ]:
downsample_hours = 2
model = ProphetDarts(
    # There are already seasonalities included by default.
    # Below is how to define some custom ones but it is actually not needed
    # add_seasonalities=[
    #     {
    #         "name": "daily_custom",  # (name of the seasonality component),
    #         "seasonal_periods": 96
    #         / (4 * downsample_hours),  # (nr of steps composing a season),
    #         "fourier_order": 3,  # (number of Fourier components to use),
    #     },
    #     {
    #         "name": "weekly_custom",  # (name of the seasonality component),
    #         "seasonal_periods": 96
    #         * 7
    #         / (4 * downsample_hours),  # (nr of steps composing a season),
    #         "fourier_order": 3,  # (number of Fourier components to use),
    #     },
    # ],
)
forecast = eval_model(
    model, series.resample(freq=f"{downsample_hours}h"), mode="statistical"
)
plot_forecast(forecast, series)

In [ ]:
deepl_kwargs = {"log_tensorboard": True}

In [ ]:
model = NBEATSModel(
    input_chunk_length=X_DIM,
    output_chunk_length=LOOKAHEAD,
    num_stacks=3,
    num_layers=2,
    layer_widths=32,
    **deepl_kwargs
)

forecast = eval_model(model, series, mode="deepl")


plot_forecast(forecast, series)

In [ ]:
model = BlockRNNModel(
    input_chunk_length=X_DIM,
    output_chunk_length=LOOKAHEAD,
    model="LSTM",
    hidden_dim=32,
    n_rnn_layers=1,
    **deepl_kwargs
)

forecast = eval_model(model, series, mode="deepl")

plot_forecast(forecast, series)

In [ ]:
model = TCNModel(
    input_chunk_length=X_DIM,
    output_chunk_length=LOOKAHEAD,
    num_filters=6,
    **deepl_kwargs
)

forecast = eval_model(model, series, mode="deepl")

plot_forecast(forecast, series)

In [ ]:
model = TransformerModel(
    input_chunk_length=X_DIM,
    output_chunk_length=LOOKAHEAD,
    dim_feedforward=64,
    nhead=2,  # default is 4
    **deepl_kwargs
)

forecast = eval_model(model, series, mode="deepl")

plot_forecast(forecast, series)

In [ ]:
model = TFTModel(
    input_chunk_length=X_DIM,
    output_chunk_length=LOOKAHEAD,
    hidden_size=16,
    **deepl_kwargs
)

forecast = eval_model(model, series, mode="deepl")

plot_forecast(forecast, series)